# COGS 109 | Member 1 | Does player size predict early NBA scoring?

**Group question:** How well do pre-draft physical measurements and athletic tests predict early NBA scoring (first-three-season PPG), and do associations vary by position?

**Proposed model:**

$$
PPG = \beta_0 + \beta_1 Height + \beta_2 Wingspan + \beta_3 Weight + \beta_4 Forward + \beta_5 Center + \epsilon
$$

*This is the proposed symbolic equation. The notebook prints the **estimated** coefficients from actual training data when run.*

**Interpretation guide:** Interpret the coefficient for `wingspan_in` as the predicted PPG change per additional inch, holding included predictors fixed. The `Forward` coefficient compares forwards with guards, the reference category, holding numeric predictors fixed. Check VIF closely: height, wingspan, and weight can be collinear.

**Sampling:** all 3 members use the same 80/20 stratified train/test split with random seed 42. The held-out test set is evaluated once; transformations and imputation use the training set only.

## 1. Load common group dataset and split
First run `00_build_dataset.ipynb`. The source file `output/nba_early_career_ppg.csv` should exist.

In [ ]:
%pip -q install pandas numpy matplotlib statsmodels scikit-learn

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display, Image
assert Path('nba_cp1_models.py').exists(), 'Place nba_cp1_models.py next to the notebook.'
from nba_cp1_models import get_split,fit_member
full, train, test = get_split()
print('Group sample:', len(full), '| Training:',len(train),'| Test:',len(test))
print('Training positions:',train.position_group.value_counts().to_dict())

## 2. Fit using training data, diagnose, then evaluate once on the held-out set
The helper also writes fitted coefficients, residual plots, Q-Q plot, scale-location plot, VIF, predictions and test scores into `output/`. The size model has an explicit **training-only** VIF rule: when VIF is above 5, remove redundant size features before the one final held-out evaluation.

In [ ]:
result = fit_member('size')
print('Slide-ready model:',result['final_formula'])
print('Test MSE:',round(result['test_mse'],3),'PPG squared')
print('Test R2:',round(result['test_r2'],3))

## 3. Inspect assumption figures (training set only)
1. **Linearity:** look for no curve in residuals versus fitted.
2. **Constant variance:** look for no fan/funnel pattern, supported by scale-location.
3. **Normal errors:** Q-Q points should roughly follow the diagonal.
4. **Independent errors:** model has one row per player; also consider differences between draft years and potential cohort correlations.
5. **No multicollinearity:** inspect VIF and numeric predictor correlation. Course rule of thumb: VIF below 5.

If any assumptions fail, describe **one meaningful issue and a tested remedy** on a separate slide. Do not claim assumptions pass without inspecting these outputs.

In [ ]:
for plot in ['residuals','qq','scale_location']:
    img=Path('output/figures/size_final_'+plot+'.png')
    if img.exists(): display(Image(filename=str(img),width=610))

vif=pd.read_csv('output/size_final_vif.csv')
display(vif)
print('Raw quantitative-feature correlations (before imputation):')
display(train[['height_in', 'wingspan_in', 'weight_lb']].corr().round(2))

## 4. Prepare your model slide
Make sure your slide includes: the final equation in math, the fitted equation with **real coefficient estimates**, a small coefficient table, numeric and qualitative interpretations, **test MSE** and **test R²**, and one sentence about all five assumption checks. Guard is the reference position. If you tried an assumption fix, show the before/after evidence separately.

In [ ]:
coef = pd.read_csv('output/size_coefficients.csv')
display(coef.round(4))
print('Final model:', result['final_formula'])
print('Training problem and fix:', result['fix'])
print('Test MSE:',result['test_mse'],'Test R2:',result['test_r2'])

## 5. Optional: held-out predictions to discuss at the expo
Do not choose example players based on training data. These are test-set players only.

In [ ]:
pred=pd.read_csv('output/size_test_predictions.csv')
pred['absolute_error']=(pred['ppg_3yr']-pred['predicted_ppg']).abs()
display(pred.sort_values('absolute_error',ascending=False).head(6).round(2))